<a href="https://colab.research.google.com/github/mro9395/risk_zones/blob/main/risk_polygon_household_counts_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple CDMX risk-polygon household counts

A minimal workflow: load the specified layers, select high-risk polygons, attach household-proxy counts from the cadastre and `lotes vulnerables`, and export one GeoJSON table. There are no segments, buffers, local windows, terrain measures, or rankings.

In [ ]:
%pip -q install geopandas pyogrio shapely pyproj rtree

from pathlib import Path
import re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
from google.colab import drive, files

DRIVE_RAW = '/content/drive/MyDrive/Dissertation/data/raw'
WORK = Path('/content/drive/MyDrive/Dissertation/data/output')
RAW = Path(DRIVE_RAW)
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive', force_remount=False)
if not RAW.exists(): raise FileNotFoundError(f'Raw-data folder not found: {RAW}')
if not WORK.exists(): raise FileNotFoundError(f'Create the output folder in Drive first: {WORK}')
METRIC_CRS = 'EPSG:32614'

RISK_FIELD = 'INTENSIDAD'
DISTRICT_FIELD = 'nomgeo'
RISK_CATEGORY_ORDER = ['baja', 'media', 'alta']  # low to high
N_HIGHEST_RISK_CATEGORIES = 1  # use 2 later to include media + alta
# File tokens and district-name aliases are intentionally separate (e.g. CUAJIMALPA vs CUAJIMALPA DE MORELOS).
CADASTRE_TOKENS = {'aob':'ALVARO_OBREGON', 'cuj':'CUAJIMALPA', 'gam':'GUSTAVO_A_MADERO', 'izp':'IZTAPALAPA'}
DISTRICT_ALIASES = {'aob':{'ALVARO OBREGON'}, 'cuj':{'CUAJIMALPA','CUAJIMALPA DE MORELOS'}, 'gam':{'GUSTAVO A MADERO'}, 'izp':{'IZTAPALAPA'}}

def norm(x):
    x = unicodedata.normalize('NFKD', str(x)).encode('ascii','ignore').decode('ascii').lower()
    return re.sub(r'[^a-z0-9]+', '_', x).strip('_')
raw_files = [p for p in RAW.rglob('*') if p.is_file()]
def matches(*tokens, extensions={'.geojson','.json','.zip'}):
    tokens=[norm(x) for x in tokens]
    # Treat catastro2021, catastro_2021, and catastro-2021 as equivalent.
    def contains(name, token):
        return token in name or token.replace('_','') in name.replace('_','')
    return [p for p in raw_files if p.suffix.lower() in extensions and all(contains(norm(p.name),t) for t in tokens)]
def one(label,*tokens):
    found=matches(*tokens)
    if len(found)!=1: raise FileNotFoundError(f'{label}: expected one match, found {found}')
    return found[0]
def read(path): return gpd.read_file(f'zip://{path}' if path.suffix.lower()=='.zip' else path)
def clean(frame,label):
    if frame.crs is None: raise ValueError(f'{label} has no CRS.')
    x=frame.to_crs(METRIC_CRS).copy(); x=x[x.geometry.notna() & ~x.geometry.is_empty]; x['geometry']=x.geometry.make_valid()
    return x[x.geometry.notna() & ~x.geometry.is_empty]


Mounted at /content/drive


## 1. Collect and preprocess the selected inputs

Only the national risk layer, district limits, four cadastres, and the supplied AOB/CUJ/IZP vulnerable-lot layers are read. GAM has no vulnerable-lots input, so its corresponding count is `NA`.

In [ ]:
risk_file=one('Risk zones','inestabilidad','laderas')
district_matches=matches('alcald')
if len(district_matches)!=1: raise FileNotFoundError(f'District file (alcaldias/alcadias): {district_matches}')
district_file=district_matches[0]
cadastre_files={c:one(f'Cadastre {c}','catastro2021',token) for c,token in CADASTRE_TOKENS.items()}
vulnerable_files={c:one(f'Vulnerable lots {c}','lotes','vulnerables',c) for c in ('aob','cuj','izp')}

# Explode multipart risk features immediately, so all downstream processing starts from single polygons.
risk=clean(read(risk_file),risk_file.name).explode(index_parts=False, ignore_index=True)
risk=risk[risk.geometry.geom_type=='Polygon'][[RISK_FIELD,'geometry']].rename(columns={RISK_FIELD:'risk_category'}).copy()
risk['risk_category']=risk.risk_category.astype(str).map(norm)
treated=set(norm(x) for x in RISK_CATEGORY_ORDER[-N_HIGHEST_RISK_CATEGORIES:])
risk=risk[risk.risk_category.isin(treated)].copy(); risk['source_risk_polygon_id']=[f'R{i:05d}' for i in range(1,len(risk)+1)]

districts=clean(read(district_file),district_file.name)[[DISTRICT_FIELD,'geometry']].rename(columns={DISTRICT_FIELD:'district_name'})
def district_code(name):
    value=norm(name)
    return next((code for code,aliases in DISTRICT_ALIASES.items() if value in {norm(a) for a in aliases}), None)
districts['district']=districts.district_name.map(district_code)
districts=districts[districts.district.notna()][['district','geometry']]
if len(districts)!=4: raise ValueError('Could not identify all four districts; inspect nomgeo values.')

# Clip high-risk polygons to study-district limits; explode every multipart feature into one Polygon per row.
risk_polygons=gpd.overlay(risk,districts,how='intersection',keep_geom_type=True)
risk_polygons=risk_polygons[~risk_polygons.geometry.is_empty].explode(index_parts=False, ignore_index=True)
risk_polygons=risk_polygons[risk_polygons.geometry.geom_type=='Polygon'].copy().reset_index(drop=True)
risk_polygons['risk_polygon_id']=[f'P{i:05d}' for i in range(1,len(risk_polygons)+1)]
risk_polygons['polygon_area_m2']=risk_polygons.area
risk_polygons['boundary_perimeter_m']=risk_polygons.length
print('Selected risk category/categories:', sorted(treated))
print('Risk-polygon records by district:', risk_polygons.groupby('district').size().to_dict())


/usr/local/lib/python3.13/dist-packages/pyogrio/raw.py:200: RuntimeWarning: Several features with id = 30 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


Selected risk category/categories: ['alta']
Risk-polygon records by district: {'aob': 624, 'cuj': 637, 'gam': 168, 'izp': 70}


## 2. Convert parcel polygons to household-proxy points

A representative point is used for each parcel. This avoids counting a parcel twice where it overlaps a risk boundary.

In [ ]:
def parcel_points(path,district):
    x=clean(read(path),path.name); x=x[x.geometry.geom_type.isin(['Polygon','MultiPolygon'])].copy()
    return gpd.GeoDataFrame({'district':district,'geometry':x.representative_point()},geometry='geometry',crs=METRIC_CRS)
cadastre=gpd.GeoDataFrame(pd.concat([parcel_points(p,d) for d,p in cadastre_files.items()],ignore_index=True),geometry='geometry',crs=METRIC_CRS)
vulnerable=gpd.GeoDataFrame(pd.concat([parcel_points(p,d) for d,p in vulnerable_files.items()],ignore_index=True),geometry='geometry',crs=METRIC_CRS)
print('Cadastre household-proxy points:',cadastre.groupby('district').size().to_dict())
print('Vulnerable-lot points:',vulnerable.groupby('district').size().to_dict())


/usr/local/lib/python3.13/dist-packages/pyogrio/raw.py:200: RuntimeWarning: OGRGeoJSONReadRawPoint(): too many members in array '[ -99.231738370429355, 19.367896087934241, 0, null ]': 4. At most 3 are handled. Ignoring extra members. Further messages of this type will be suppressed.
  return ogr_read(


Cadastre household-proxy points: {'aob': 96448, 'cuj': 21786, 'gam': 163195, 'izp': 236303}
Vulnerable-lot points: {'aob': 5316, 'cuj': 3107, 'izp': 4957}


## 3–5. Intersect polygons with both sources, count all households inside, and export

`catastro_total_households_inside_risk_polygon` counts every cadastre representative point inside the complete risk polygon. It is not restricted to a distance band. `lotes_vulnerables_total_parcels_inside_risk_polygon` is the corresponding government-selected parcel count.

In [ ]:
cat_index, vulnerable_index = cadastre.sindex, vulnerable.sindex
def count_points(points,index,polygon,district):
    ids=list(index.query(polygon,predicate='intersects'))
    if not ids: return 0
    x=points.iloc[ids]; return int(((x.district==district) & x.geometry.within(polygon)).sum())

cat_counts=[]; vulnerable_counts=[]
for row in risk_polygons.itertuples():
    cat_counts.append(count_points(cadastre,cat_index,row.geometry,row.district))
    vulnerable_counts.append(count_points(vulnerable,vulnerable_index,row.geometry,row.district) if row.district in vulnerable_files else np.nan)
risk_polygons['catastro_total_households_inside_risk_polygon']=cat_counts
risk_polygons['lotes_vulnerables_total_parcels_inside_risk_polygon']=vulnerable_counts

# GeoJSON requires WGS84 longitude/latitude coordinates. Its attribute table is the requested geodataset table.
out=risk_polygons.to_crs(4326)
out.to_file(WORK/'risk_polygon_household_counts.geojson',driver='GeoJSON')
cadastre.to_crs(4326).to_file(WORK/'catastro_points.geojson',driver='GeoJSON')
vulnerable.to_crs(4326).to_file(WORK/'lotes_vulnerables_points.geojson',driver='GeoJSON')

manifest=pd.DataFrame([('risk_zones',risk_file.name),('district_limits',district_file.name),*[(f'catastro_{d}',p.name) for d,p in cadastre_files.items()],*[(f'lotes_vulnerables_{d}',p.name) for d,p in vulnerable_files.items()]],columns=['role','file'])
manifest.to_csv(WORK/'inputs_used_manifest.csv',index=False)

## Output fields

- `risk_polygon_id`: source high-risk polygon identifier created for this run.
- `district`, `risk_category`, `polygon_area_m2`: polygon context.
- `catastro_total_households_inside_risk_polygon`: complete-polygon household proxy count.
- `lotes_vulnerables_total_parcels_inside_risk_polygon`: complete-polygon selected vulnerable-parcel count; `NA` in GAM because its source file was not supplied.

In [ ]:
cat_index, vulnerable_index = cadastre.sindex, vulnerable.sindex
def count_points(points,index,polygon,district):
    ids=list(index.query(polygon,predicate='intersects'))
    if not ids: return 0
    x=points.iloc[ids]; return int(((x.district==district) & x.geometry.within(polygon)).sum())

cat_counts=[]; vulnerable_counts=[]
for row in risk_polygons.itertuples():
    cat_counts.append(count_points(cadastre,cat_index,row.geometry,row.district))
    vulnerable_counts.append(count_points(vulnerable,vulnerable_index,row.geometry,row.district) if row.district in vulnerable_files else np.nan)
risk_polygons['catastro_total_households_inside_risk_polygon']=cat_counts
risk_polygons['lotes_vulnerables_total_parcels_inside_risk_polygon']=vulnerable_counts

# GeoJSON requires WGS84 longitude/latitude coordinates. Its attribute table is the requested geodataset table.
out=risk_polygons.to_crs(4326)
out.to_file(WORK/'risk_polygon_household_counts.geojson',driver='GeoJSON')
cadastre.to_crs(4326).to_file(WORK/'catastro_points.geojson',driver='GeoJSON')
vulnerable.to_crs(4326).to_file(WORK/'lotes_vulnerables_points.geojson',driver='GeoJSON')

manifest=pd.DataFrame([('risk_zones',risk_file.name),('district_limits',district_file.name),*[(f'catastro_{d}',p.name) for d,p in cadastre_files.items()],*[(f'lotes_vulnerables_{d}',p.name) for d,p in vulnerable_files.items()]],columns=['role','file'])
manifest.to_csv(WORK/'inputs_used_manifest.csv',index=False)

In [ ]:
# Read and concatenate the original cadastre polygon files
cadastre_polygons_list = []
for district_code, file_path in cadastre_files.items():
    # The 'clean' function expects a label for errors, using file_path.name
    gdf = clean(read(file_path), file_path.name)
    cadastre_polygons_list.append(gdf)

cadastre_polygons = pd.concat(cadastre_polygons_list, ignore_index=True)

# Export the original cadastre parcel polygons in WGS84 (EPSG:4326).
catastro_parcels_4326 = cadastre_polygons.to_crs(4326)
catastro_parcels_4326.to_file(WORK/'catastro_parcels_4326.geojson', driver='GeoJSON')
print('Exported', f'{len(catastro_parcels_4326):,}', 'cadastre parcels to', WORK / 'catastro_parcels_4326.geojson')

Exported 517,740 cadastre parcels to /content/drive/MyDrive/Dissertation/data/output/catastro_parcels_4326.geojson
